# Mobil Yüz Rekonstrüksiyon — Kaggle Eğitim Notebook'u

**GPU:** Kaggle P100/T4 (haftada 30 saat ücretsiz)  
**Çıktılar:** `/kaggle/working/` — session sonunda zip olarak indir  
**Kalıcı checkpoint:** Kaggle Dataset olarak kaydet (aşağıda açıklanıyor)

---
### Adımlar
1. Kurulum
2. Kodu GitHub'dan çek
3. Config (Kaggle yolları)
4. Veri seti hazırlama
5. Ön işleme
6. Eğitim
7. TFLite export
8. Checkpoint'i kalıcı kaydet

## 1. Kurulum

In [ ]:
import subprocess, sys

!nvidia-smi

!pip install -q \
    insightface \
    onnxruntime-gpu \
    facenet-pytorch \
    pytorch-msssim \
    torchmetrics \
    opencv-python-headless \
    onnx \
    tensorboard \
    ai-edge-torch

print('Kurulum tamamlandı.')

## 2. Proje Kodunu GitHub'dan Çek

In [ ]:
import os, sys

!git clone https://github.com/BuhraOzdemir/face-recon.git /kaggle/working/face-recon
sys.path.insert(0, '/kaggle/working/face-recon')
%cd /kaggle/working/face-recon

print('Proje kodu hazır:', os.getcwd())

## 3. Config (Kaggle Yolları)

Kaggle'da klasör yapısı:
```
/kaggle/input/DATASET_ADI/   ← eklediğin veri seti
/kaggle/working/             ← tüm çıktılar (checkpoint, export, log)
```

In [ ]:
from src.config import Config, DataConfig, ModelConfig, LossConfig, TrainConfig, ExportConfig

# Kaggle'a eklediğin veri setinin adını buraya yaz
# Örnek: vggface2 dataseti eklediysen -> /kaggle/input/vggface2/
DATASET_NAME = 'vggface2'  # <-- değiştir

cfg = Config(
    data=DataConfig(
        data_dir      = f'/kaggle/input/{DATASET_NAME}',
        processed_dir = '/kaggle/working/processed',
        image_size    = 128,
        val_split     = 0.05,
        num_workers   = 2,
    ),
    model=ModelConfig(
        embedding_dim    = 512,
        initial_spatial  = 4,
        initial_channels = 128,
        decoder_channels = (128, 128, 64, 32, 16),
    ),
    loss=LossConfig(
        phase1_epochs      = 10,
        phase2_epochs      = 50,
        phase3_epochs      = 40,
        vgg_layer          = 'relu3_3',
        facenet_input_size = 160,
    ),
    train=TrainConfig(
        epochs            = 100,
        batch_size        = 64,
        learning_rate     = 1e-4,
        weight_decay      = 1e-4,
        warmup_epochs     = 5,
        save_dir          = '/kaggle/working/checkpoints',
        save_every_epochs = 5,
        keep_last_n       = 3,
        patience          = 10,
        use_amp           = True,
        log_dir           = '/kaggle/working/logs',
        log_every_steps   = 50,
    ),
    export=ExportConfig(
        export_dir  = '/kaggle/working/export',
        model_name  = 'face_decoder',
        quantize_int8 = True,
    ),
)

print(f'Toplam epoch : {cfg.total_epochs()}')
print(f'Veri giriş   : {cfg.data.data_dir}')
print(f'Checkpoint   : {cfg.train.save_dir}')
print(f'Export       : {cfg.export.export_dir}')

## 4. Veri Seti Ekleme

Kaggle'da veri seti eklemenin iki yolu:

**Yol A — Hazır Kaggle dataseti (önerilir):**
1. Notebook'un sağ panelinde **"+ Add data"** butonuna tıkla
2. Arama kutusuna `vggface2` veya `ms1m` yaz
3. Dataseti ekle → `/kaggle/input/DATASET_ADI/` olarak bağlanır
4. Yukarıdaki `DATASET_NAME` değişkenini güncelle

**Yol B — Kendi verinle:**
1. Kaggle'da yeni bir dataset oluştur
2. Zip'leyerek yükle
3. Notebook'a ekle

**Beklenen klasör yapısı:**
```
/kaggle/input/vggface2/
    n000002/
        0001_01.jpg
    n000003/
        ...
```

In [ ]:
import os
from pathlib import Path

data_path = Path(cfg.data.data_dir)

if data_path.exists():
    id_dirs = [d for d in data_path.iterdir() if d.is_dir()]
    total_imgs = sum(len(list(d.glob('*.jpg')) + list(d.glob('*.png'))) for d in id_dirs[:100])
    print(f'Veri seti bulundu: {data_path}')
    print(f'Kimlik sayisi    : {len(id_dirs)}')
    print(f'Ortalama görüntü : ~{total_imgs // min(100, len(id_dirs))} per kimlik')
else:
    print(f'UYARI: {data_path} bulunamadı!')
    print('Lütfen sağ panelden veri setini ekle ve DATASET_NAME değişkenini güncelle.')
    
    # Demo için sentetik veri
    print('\nDemo modu: sentetik veri oluşturuluyor...')
    import numpy as np
    from PIL import Image
    raw_dir = '/kaggle/working/demo_data'
    os.makedirs(raw_dir, exist_ok=True)
    for i in range(50):
        id_dir = os.path.join(raw_dir, f'identity_{i:04d}')
        os.makedirs(id_dir, exist_ok=True)
        for j in range(20):
            img = Image.fromarray(np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8))
            img.save(os.path.join(id_dir, f'img_{j:04d}.jpg'))
    cfg.data.data_dir = raw_dir
    print(f'Demo veri: {raw_dir}')

## 5. Ön İşleme

In [ ]:
# Daha önce işlediysen ve /kaggle/working/processed varsa atla
manifest_path = '/kaggle/working/processed/manifest.txt'

if os.path.exists(manifest_path):
    print(f'İşlenmiş veri zaten mevcut: {manifest_path}')
    with open(manifest_path) as f:
        n = sum(1 for l in f if l.strip())
    print(f'Örnek sayısı: {n:,}')
else:
    from src.data.preprocess import preprocess_dataset
    manifest_path = preprocess_dataset(
        raw_dir     = cfg.data.data_dir,
        out_dir     = cfg.data.processed_dir,
        output_size = cfg.data.image_size,
        max_per_id  = 100,
    )

In [ ]:
from src.data.dataset import build_dataloaders
import torch

train_loader, val_loader = build_dataloaders(
    manifest_path = manifest_path,
    image_size    = cfg.data.image_size,
    batch_size    = cfg.train.batch_size,
    val_split     = cfg.data.val_split,
    num_workers   = cfg.data.num_workers,
)
print(f'Train: {len(train_loader.dataset):,}  |  Val: {len(val_loader.dataset):,}')

emb, img = next(iter(train_loader))
print(f'Embedding: {emb.shape}  |  Görüntü: {img.shape}  |  Aralık: [{img.min():.2f}, {img.max():.2f}]')

## 6. Eğitim

In [ ]:
from src.train import train

# Önceki session'dan checkpoint varsa devam et
resume_path = '/kaggle/working/checkpoints/best_model.pt'
if not os.path.exists(resume_path):
    resume_path = None

if resume_path:
    print(f'Checkpoint bulundu, devam ediliyor: {resume_path}')
else:
    print('Sıfırdan eğitim başlıyor...')

best_ckpt = train(
    cfg           = cfg,
    manifest_path = manifest_path,
    resume_from   = resume_path,
)
print(f'Eğitim tamamlandı: {best_ckpt}')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/logs

## 7. Sonuçları Görselleştir

In [ ]:
import torch, matplotlib.pyplot as plt
from src.models.decoder import FaceDecoder
from src.data.dataset import denormalize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = FaceDecoder(
    embedding_dim    = cfg.model.embedding_dim,
    initial_spatial  = cfg.model.initial_spatial,
    initial_channels = cfg.model.initial_channels,
    decoder_channels = cfg.model.decoder_channels,
).to(device)

state = torch.load(best_ckpt, map_location=device)
model.load_state_dict(state['model'])
model.eval()

val_embs, val_imgs = next(iter(val_loader))
with torch.no_grad():
    generated = model(val_embs[:8].to(device)).cpu()

real_np = denormalize(val_imgs[:8]).permute(0,2,3,1).numpy()
gen_np  = denormalize(generated).permute(0,2,3,1).numpy()

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i in range(8):
    axes[0,i].imshow(real_np[i].clip(0,1)); axes[0,i].set_title('Gerçek', fontsize=8); axes[0,i].axis('off')
    axes[1,i].imshow(gen_np[i].clip(0,1));  axes[1,i].set_title('Üretilen', fontsize=8); axes[1,i].axis('off')
plt.suptitle('Gerçek vs Üretilen', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/results_sample.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. TFLite Export

In [ ]:
from src.export_tflite import export_model

exported = export_model(
    checkpoint_path = best_ckpt,
    export_dir      = cfg.export.export_dir,
    cfg             = cfg,
)

print('\nExport tamamlandı:')
for fmt, path in exported.items():
    size_mb = os.path.getsize(path) / 1e6
    print(f'  {fmt:12s}: {size_mb:.2f} MB  →  {path}')

## 9. Checkpoint'i Kalıcı Kaydet

Kaggle session kapandığında `/kaggle/working/` silinir.  
Checkpoint'i **Kaggle Dataset** olarak kaydetmek için:

In [ ]:
# Önemli dosyaları tek zip'e topla
import zipfile, shutil

zip_path = '/kaggle/working/face_recon_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Best model checkpoint
    if os.path.exists('/kaggle/working/checkpoints/best_model.pt'):
        zf.write('/kaggle/working/checkpoints/best_model.pt', 'best_model.pt')
    # TFLite dosyaları
    for fmt, path in exported.items():
        if os.path.exists(path):
            zf.write(path, os.path.basename(path))
    # Örnek görüntüler
    if os.path.exists('/kaggle/working/results_sample.png'):
        zf.write('/kaggle/working/results_sample.png', 'results_sample.png')

size_mb = os.path.getsize(zip_path) / 1e6
print(f'Zip oluşturuldu: {zip_path}  ({size_mb:.1f} MB)')
print()
print('Kalıcı kaydetmek için:')
print('  1. Sağ panelde Output sekmesine tıkla')
print('  2. face_recon_results.zip yanındaki üç noktaya tıkla')
print('  3. "Save to Dataset" seç → yeni dataset oluştur')
print('  4. Sonraki session\'da bu dataseti notebook\'a ekle')

In [ ]:
# Sonraki session'da checkpoint'i geri yüklemek için:
# 1. Kaggle dataset'ini notebook'a ekle
# 2. Aşağıdaki kodu çalıştır:

# import shutil, zipfile
# with zipfile.ZipFile('/kaggle/input/DATASET_ADIN/face_recon_results.zip') as zf:
#     zf.extract('best_model.pt', '/kaggle/working/checkpoints/')
# print('Checkpoint geri yüklendi.')